In [1]:
!pip install -q --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.1 MB/s eta 0:00:0000:010:01


In [2]:
!pip install -q transformers datasets accelerate peft evaluate jiwer librosa soundfile

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.

In [3]:
!pip install ijson

In [4]:
import os
import json
import random
import torch
import librosa
import numpy as np
from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

2025-12-14 15:47:26.102450: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765727246.124193     115 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765727246.130728     115 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
DATA_DIR = "/kaggle/input/weight-vietasr/train_full_manifest.jsonl"
MANIFEST_PATH = os.path.join(DATA_DIR)

def load_manifest(manifest_path):
    entries = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            entries.append({"audio_path": item["audio"], "sentence": item["text"]})
    print(f"✅ Loaded {len(entries)} samples")
    return entries

all_data = load_manifest(MANIFEST_PATH)

✅ Loaded 53997 samples


In [6]:
VAL_RATIO = 0.1
random.seed(42)
random.shuffle(all_data)
val_size = int(len(all_data) * VAL_RATIO)
train_entries = all_data[val_size:]
val_entries = all_data[:val_size]

print(f"Train: {len(train_entries)} samples")
print(f"Val: {len(val_entries)} samples")

Train: 48598 samples
Val: 5399 samples


# Train batch 8-10

In [9]:
#Tiep tuc train batch 8
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

MODEL_DIR = "/kaggle/input/batch/other/default/1"

processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("✅ Loaded checkpoint batch 8")


✅ Loaded checkpoint batch 5


In [10]:
MAX_LABEL_LEN = 448   # giới hạn Whisper

def prepare_dataset(batch):
    audio_path = batch["audio_path"]

    try:
        audio_array, _ = librosa.load(audio_path, sr=16000)
    except:
        return None  # bỏ sample lỗi

    # Encode audio
    input_features = processor(
        audio_array,
        sampling_rate=16000
    ).input_features[0]

    # Encode text
    labels = processor.tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LABEL_LEN
    ).input_ids

    # ❌ BỎ SAMPLE QUÁ DÀI
    if len(labels) > MAX_LABEL_LEN:
        return None

    batch["input_features"] = input_features
    batch["labels"] = labels
    return batch

In [11]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch


In [12]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
metric = evaluate.load("wer")

In [13]:
 def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [14]:
BATCH_SIZE = 5000
train_batches = [train_entries[i:i+BATCH_SIZE] for i in range(0, len(train_entries), BATCH_SIZE)]
print(f"Chia thành {len(train_batches)} batches (~{BATCH_SIZE} samples/batch)")

Chia thành 10 batches (~5000 samples/batch)


In [15]:
val_dataset = Dataset.from_list(val_entries)

# Map 1 lần duy nhất
val_dataset = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    desc="Preprocessing validation dataset"
)
print(f"✅ Validation dataset ready: {len(val_dataset)} samples")

Preprocessing validation dataset:   0%|          | 0/5399 [00:00<?, ? examples/s]

✅ Validation dataset ready: 5399 samples


In [16]:
    LAST_FINISHED_BATCH = 7  # đã train xong batch 1 → 5
    
    for idx in range(LAST_FINISHED_BATCH, len(train_batches)):
        batch = train_batches[idx]
        print(f"\n🚀 Training batch {idx+1}/{len(train_batches)}")
    
        train_dataset = Dataset.from_list(batch)
    
        train_dataset = train_dataset.map(
            prepare_dataset,
            remove_columns=train_dataset.column_names,
            desc=f"Preprocessing batch {idx+1}"
        )
    
        train_dataset = train_dataset.filter(lambda x: x is not None)
    
        training_args = Seq2SeqTrainingArguments(
            output_dir="./whisper-train",
    
            eval_strategy="no",
            do_eval=False,
    
            per_device_train_batch_size=1,
            gradient_accumulation_steps=16,
            learning_rate=1e-5,
            num_train_epochs=3,
    
            fp16=True,
            logging_steps=50,
            save_steps=500,
            save_total_limit=2,
    
            report_to="none",
            remove_unused_columns=False,
        )
    
        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            processing_class=processor,
        )
    
        trainer.train()
    
        # 🔐 Lưu checkpoint batch mới
        batch_output_dir = f"/kaggle/working/batch_{idx+1}"
        model.save_pretrained(batch_output_dir)
        processor.save_pretrained(batch_output_dir)
    
        del trainer, train_dataset
        import gc; gc.collect()
        torch.cuda.empty_cache()
    
        print(f"✅ Finished batch {idx+1}")


🚀 Training batch 8/10


Preprocessing batch 8:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
50,1.416500
100,0.618200
150,0.590600
200,0.558600
250,0.526000
300,0.581200
350,0.447100
400,0.405200
450,0.404900
500,0.395500


...

✅ Finished batch 9

🚀 Training batch 10/10


Preprocessing batch 10:   0%|          | 0/3598 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3598 [00:00<?, ? examples/s]

Step,Training Loss
50,0.548600
100,0.515900
150,0.537000
200,0.526700
250,0.420600
300,0.375300
350,0.385500
400,0.365200
450,0.348800
500,0.277300


✅ Finished batch 10


In [17]:
import os
import zipfile

BASE_DIR = "/kaggle/working"
ZIP_PATH = "/kaggle/working/whisper_batches.zip"

# Tìm tất cả thư mục batch_x
batch_dirs = sorted([
    d for d in os.listdir(BASE_DIR)
    if d.startswith("batch_") and os.path.isdir(os.path.join(BASE_DIR, d))
])

print("📂 Found batches:", batch_dirs)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for batch_dir in batch_dirs:
        batch_path = os.path.join(BASE_DIR, batch_dir)
        for root, _, files in os.walk(batch_path):
            for file in files:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, BASE_DIR)
                zipf.write(full_path, arcname)

print(f"\n✅ Done! ZIP saved at: {ZIP_PATH}")

📂 Found batches: ['batch_10', 'batch_8', 'batch_9']

✅ Done! ZIP saved at: /kaggle/working/whisper_batches.zip


# Train batch 1-7

In [ ]:
MODEL_NAME = "openai/whisper-base"
OUTPUT_DIR = "/kaggle/working/whisper-base-ft-batch"
os.makedirs(OUTPUT_DIR, exist_ok=True)

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Vietnamese", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="vi", task="transcribe")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
MAX_LABEL_LEN = 448   # giới hạn Whisper

def prepare_dataset(batch):
    audio_path = batch["audio_path"]

    try:
        audio_array, _ = librosa.load(audio_path, sr=16000)
    except:
        return None  # bỏ sample lỗi

    # Encode audio
    input_features = processor(
        audio_array,
        sampling_rate=16000
    ).input_features[0]

    # Encode text
    labels = processor.tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LABEL_LEN
    ).input_ids

    # ❌ BỎ SAMPLE QUÁ DÀI
    if len(labels) > MAX_LABEL_LEN:
        return None

    batch["input_features"] = input_features
    batch["labels"] = labels
    return batch

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
metric = evaluate.load("wer")

In [ ]:
 def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
BATCH_SIZE = 5000
train_batches = [train_entries[i:i+BATCH_SIZE] for i in range(0, len(train_entries), BATCH_SIZE)]
print(f"Chia thành {len(train_batches)} batches (~{BATCH_SIZE} samples/batch)")

In [ ]:
val_dataset = Dataset.from_list(val_entries)

# Map 1 lần duy nhất
val_dataset = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    desc="Preprocessing validation dataset"
)
print(f"✅ Validation dataset ready: {len(val_dataset)} samples")

In [ ]:
import shutil
import gc
import torch

for idx, batch in enumerate(train_batches):
    print(f"\n🚀 Training batch {idx+1}/{len(train_batches)}")

    train_dataset = Dataset.from_list(batch)

    train_dataset = train_dataset.map(
        prepare_dataset,
        remove_columns=train_dataset.column_names,
        desc=f"Preprocessing batch {idx+1}"
    )

    train_dataset = train_dataset.filter(lambda x: x is not None)

    # ===== Thư mục checkpoint =====
    batch_output_dir = os.path.join(OUTPUT_DIR, f"batch_{idx+1}")
    os.makedirs(batch_output_dir, exist_ok=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=batch_output_dir,     # ⚠️ cho trùng checkpoint
        eval_strategy="no",
        do_eval=False,

        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=1e-5,
        num_train_epochs=3,

        fp16=True,
        logging_steps=50,

        save_strategy="no",               # ⚠️ tắt save của Trainer
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        processing_class=processor,
    )

    trainer.train()

    # ===== SAVE CHECKPOINT =====
    model.save_pretrained(batch_output_dir)
    processor.save_pretrained(batch_output_dir)

    # ===== ZIP CHECKPOINT =====
    zip_path = f"/kaggle/working/batch_{idx+1}.zip"
    shutil.make_archive(
        base_name=zip_path.replace(".zip", ""),
        format="zip",
        root_dir=batch_output_dir
    )

    print(f"📦 Zipped checkpoint: {zip_path}")

    # ===== CLEANUP =====
    del train_dataset, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"✅ Finished batch {idx+1}")

# Danh gia model

In [ ]:
MAX_LABEL_LEN = 448

def prepare_dataset(batch):
    audio_path = batch["audio_path"]

    try:
        audio_array, _ = librosa.load(audio_path, sr=16000)
    except:
        return None

    input_features = processor(
        audio_array,
        sampling_rate=16000
    ).input_features[0]

    labels = processor.tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LABEL_LEN
    ).input_ids

    if len(labels) > MAX_LABEL_LEN:
        return None

    batch["input_features"] = input_features
    batch["labels"] = labels
    return batch

In [ ]:
MODEL_DIR = "/kaggle/input/batch-train/other/train/1/batch_10"

processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [ ]:
val_dataset = Dataset.from_list(val_entries)

# Map 1 lần duy nhất
val_dataset = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    desc="Preprocessing validation dataset"
)
print(f"✅ Validation dataset ready: {len(val_dataset)} samples")

In [ ]:
TEST_DIR = "/kaggle/input/weight-vietasr/test_manifest.jsonl"
def load_test_manifest(TEST_DIR):
    entries = []
    with open(TEST_DIR, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            entries.append({
                "audio_path": item["audio"],
                "sentence": item["text"]
            })
    return entries
    
test_entries = load_test_manifest(test_dataset)

test_dataset = Dataset.from_list(test_entries)

test_dataset = test_dataset.map(
    prepare_dataset,
    remove_columns=test_dataset.column_names,
    desc="Preprocessing TEST dataset"
)

test_dataset = test_dataset.filter(lambda x: x is not None)

print(f"✅ Test samples: {len(test_dataset)}")

In [ ]:
wer_metric = evaluate.load("wer")

predictions, references = [], []

for sample in tqdm(val_dataset):
    input_features = sample["input_features"].unsqueeze(0).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=256
        )

    pred_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    label_ids = sample["labels"]
    label_ids = [
        i if i != -100 else processor.tokenizer.pad_token_id
        for i in label_ids
    ]

    ref_text = processor.decode(label_ids, skip_special_tokens=True)

    predictions.append(pred_text)
    references.append(ref_text)

wer = wer_metric.compute(predictions=predictions, references=references)

print(f"🎯 FINAL WER Val (batch_10): {wer * 100:.2f}%")

In [ ]:
wer_metric = evaluate.load("wer")

predictions, references = [], []

for sample in tqdm(test_dataset):
    input_features = sample["input_features"].unsqueeze(0).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=256
        )

    pred_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    label_ids = sample["labels"]
    label_ids = [
        i if i != -100 else processor.tokenizer.pad_token_id
        for i in label_ids
    ]

    ref_text = processor.decode(label_ids, skip_special_tokens=True)

    predictions.append(pred_text)
    references.append(ref_text)

wer = wer_metric.compute(predictions=predictions, references=references)

print(f"🎯 FINAL WER Test (batch_10): {wer * 100:.2f}%")

# DEMO

In [4]:
!pip install gradio torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.9 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.4
    Uninstalling pydantic-2.12.4:
      Successfully uninstalled pydantic-2.12.4m━━━━━━━━━━━━━━━━━━━ 1/2 [pydantic]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pydantic]1/2 [pydantic]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [28]:
import gradio as gr
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import numpy as np

In [ ]:
model_name = "/kaggle/input/batch-train/other/train/1/batch_10"  # Hoặc path local của bạn

print("🔄 Đang load model...")
try:
    # Load processor và model Whisper
    processor = WhisperProcessor.from_pretrained(model_path)
    model = WhisperForConditionalGeneration.from_pretrained(model_path)
    
    # Chuyển model sang GPU nếu có
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    print(f"✅ Load model thành công! Đang chạy trên: {device}")
except Exception as e:
    print(f"❌ Lỗi khi load model: {e}")
    print("💡 Sử dụng model Whisper mặc định từ OpenAI")
    # Fallback sang model Whisper mặc định
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

# ==============================================================================
# HÀM XỬ LÝ ÂM THANH VÀ CHUYỂN ĐỔI THÀNH TEXT
# ==============================================================================

def transcribe_audio(audio_input):
    """
    Nhận dạng giọng nói từ audio và chuyển thành text tiếng Việt
    
    Args:
        audio_input: tuple (sample_rate, audio_array) từ Gradio
    
    Returns:
        text: văn bản tiếng Việt được nhận dạng
    """
    try:
        # Kiểm tra input
        if audio_input is None:
            return "❌ Vui lòng ghi âm hoặc upload file audio"
        
        # Gradio trả về tuple (sample_rate, audio_data) hoặc đường dẫn file
        if isinstance(audio_input, str):
            # Nếu là đường dẫn file
            audio_array, sample_rate = librosa.load(audio_input, sr=16000)
        else:
            # Nếu là tuple từ microphone
            sample_rate, audio_array = audio_input
            # Resample về 16kHz nếu cần
            if sample_rate != 16000:
                audio_array = librosa.resample(
                    y=audio_array.astype(np.float32),
                    orig_sr=sample_rate,
                    target_sr=16000
                )
        
        # Chuyển sang mono nếu là stereo
        if len(audio_array.shape) > 1:
            audio_array = np.mean(audio_array, axis=1)
        
        # Chuẩn hóa audio
        audio_array = audio_array.astype(np.float32)
        
        # Xử lý với processor
        input_features = processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features
        
        input_features = input_features.to(device)
        
        # Generate token ids với forced_decoder_ids để output tiếng Việt
        forced_decoder_ids = processor.get_decoder_prompt_ids(
            language="vi",
            task="transcribe"
        )
        
        # Inference
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids
            )
        
        # Decode prediction
        transcription = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]
        
        # Trả về kết quả
        if transcription.strip() == "":
            return "⚠️ Không nhận dạng được giọng nói. Vui lòng thử lại với âm thanh rõ ràng hơn."
        
        return f"📝 Văn bản nhận dạng:\n\n{transcription}"
    
    except Exception as e:
        return f"❌ Lỗi xảy ra: {str(e)}\n\nVui lòng thử lại hoặc kiểm tra file audio."

# ==============================================================================
# TẠO GIAO DIỆN GRADIO
# ==============================================================================

# Custom CSS để làm đẹp giao diện
custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
}
.output-text {
    font-size: 18px;
    line-height: 1.6;
}
"""

# Tạo interface
with gr.Blocks(css=custom_css, title="Speech to Text Tiếng Việt - Whisper") as demo:
    gr.Markdown(
        """
        # 🎤 Chuyển đổi Giọng nói thành Văn bản Tiếng Việt
        ### Powered by Whisper Model
        
        Ghi âm giọng nói của bạn hoặc upload file audio, hệ thống sẽ tự động chuyển đổi thành văn bản tiếng Việt.
        
        ### 📌 Hướng dẫn sử dụng:
        1. **Ghi âm trực tiếp**: Click vào nút microphone để ghi âm
        2. **Upload file**: Hoặc upload file audio có sẵn (hỗ trợ .wav, .mp3, .flac, .m4a)
        3. Nhấn **"Chuyển đổi"** để nhận dạng giọng nói
        
        ### ⚡ Lưu ý:
        - Nói rõ ràng, tốc độ vừa phải
        - Giảm thiểu tiếng ồn xung quanh
        - File audio nên dưới 30 giây để kết quả tốt nhất
        """
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            # Input audio với cả microphone và upload file
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎙️ Ghi âm hoặc Upload Audio"
            )
            
            # Nút submit
            submit_btn = gr.Button(
                "🚀 Chuyển đổi sang Text", 
                variant="primary",
                size="lg"
            )
            
            # Nút xóa
            clear_btn = gr.Button("🗑️ Xóa", size="sm")
        
        with gr.Column(scale=1):
            # Output text
            output_text = gr.Textbox(
                label="📄 Kết quả nhận dạng",
                placeholder="Văn bản sẽ hiển thị ở đây...",
                lines=10,
                elem_classes="output-text"
            )
    
    gr.Markdown(
        """
        ---
        ### 💡 Tips để có kết quả tốt nhất:
        - ✅ Môi trường yên tĩnh
        - ✅ Phát âm rõ ràng
        - ✅ Giọng nói tự nhiên
        - ✅ Tránh nói quá nhanh hoặc quá chậm
        
        ### 🔧 Thông tin kỹ thuật:
        - Model: Whisper (fine-tuned cho tiếng Việt)
        - Sampling rate: 16kHz
        - Device: {}
        - Language: Vietnamese
        """.format("GPU 🚀" if torch.cuda.is_available() else "CPU 💻")
    )
    
    # Kết nối các sự kiện
    submit_btn.click(
        fn=transcribe_audio,
        inputs=audio_input,
        outputs=output_text
    )
    
    clear_btn.click(
        fn=lambda: (None, ""),
        outputs=[audio_input, output_text]
    )

# ==============================================================================
# KHỞI CHẠY DEMO
# ==============================================================================

if __name__ == "__main__":
    # Launch với cấu hình phù hợp cho Kaggle
    demo.launch(
        share=True,  # Tạo public link
        debug=True,  # Hiển thị lỗi chi tiết
        show_error=True
    )

🔄 Đang load model...
❌ Lỗi khi load model: name 'model_path' is not defined
💡 Sử dụng model Whisper mặc định từ OpenAI


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://f1888498ee289b3100.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
import librosa

def transcribe_wav(
    wav_path,
    max_new_tokens=256
):
    # Load audio
    audio, sr = librosa.load(wav_path, sr=16000)

    # Extract features
    input_features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(DEVICE)

    # Inference
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=max_new_tokens
        )

    # Decode
    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return transcription

In [22]:
demo_wav = "/kaggle/input/data-speech-to-text/Audio_logs_vie_wav/Audio_logs_vie_wav/Audio_logs_vie_31.wav"  # 🔁 đổi path nếu cần

text = transcribe_wav(demo_wav)

print("🎤 AUDIO:", demo_wav)
print("📝 TRANSCRIPTION:")
print(text)


🎤 AUDIO: /kaggle/input/data-speech-to-text/Audio_logs_vie_wav/Audio_logs_vie_wav/Audio_logs_vie_31.wav
📝 TRANSCRIPTION:
ai trong du lịch cá nhân hóa hành trình dựa trên sản thích khách hàng.
